# Face Swap Training — Kaggle Notebook

**Before running:**
1. Add dataset: `vggface2-dataset` (Kaggle Datasets tab -> Add Data)
2. Add dataset: `deca-pretrained-models` (upload your DECA data/ folder as a Kaggle dataset)
3. Enable GPU: Settings -> Accelerator -> GPU T4 x2 (or P100)
4. Set `MAX_IDENTITIES` below to control how much of VGGFace2 to use

**Dataset on Kaggle:**
- VGGFace2: search 'vggface2' on kaggle.com/datasets
- DECA data: upload `data/` folder (deca_model.tar, generic_model.pkl, landmark_embedding.npy)

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MAX_IDENTITIES   = 1000      # use 1000 out of 9131 identities (~360K images)
BATCH_SIZE       = 8         # fits comfortably on T4 16GB
PHASE1_EPOCHS    = 5         # reconstruction warmup
PHASE2_EPOCHS    = 10        # full face swap training
LR               = 1e-4
TOKEN_DIM        = 512
CHECKPOINT_DIR   = "/kaggle/working/checkpoints"
LOG_EVERY        = 50        # log every N batches
SAVE_EVERY       = 1         # save checkpoint every N epochs

# Paths (adjust if your dataset is mounted differently)
VGGFACE2_ROOT    = "/kaggle/input/vggface2/data/vggface2_train/train"
DECA_ROOT        = "/kaggle/input/deca-pretrained-models"
DECA_REPO        = "/kaggle/working/DECA"     # cloned below

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Config OK')

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
# Kaggle has torch/torchvision pre-installed. We only need extras.
!pip install -q transformers diffusers accelerate timm scikit-image kornia yacs imageio

In [ ]:
# ── Clone / copy project code ──────────────────────────────────────────────
# Option A: clone from GitHub (if your repo is public)
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /kaggle/working/DECA

# Option B: upload your code as a Kaggle dataset and copy it
# !cp -r /kaggle/input/your-code-dataset/* /kaggle/working/DECA/

# For now we assume code is at DECA_REPO
import sys
sys.path.insert(0, DECA_REPO)
sys.path.insert(0, os.path.join(DECA_REPO, 'face_swap'))
print('Paths set')

In [ ]:
# ── Install chumpy (DECA dependency, has Python 3 compat issues) ───────────
!pip install -q chumpy --no-build-isolation --no-deps

# Patch chumpy numpy compat
import pathlib
chumpy_init = pathlib.Path('/usr/local/lib/python3.10/dist-packages/chumpy/__init__.py')
if chumpy_init.exists():
    txt = chumpy_init.read_text()
    txt = txt.replace(
        'from numpy import bool, int, float, complex, object, unicode, str, nan, inf',
        'from numpy import nan, inf'
    )
    chumpy_init.write_text(txt)
    print('chumpy patched')

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import gc
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from PIL import Image
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32
print(f'Device: {DEVICE}  |  dtype: {DTYPE}')
print(f'GPU: {torch.cuda.get_device_name(0) if DEVICE=="cuda" else "none"}')

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────
from data.vggface2_dataset import VGGFace2PairDataset, collate_fn

dataset = VGGFace2PairDataset(
    root=VGGFACE2_ROOT,
    max_identities=MAX_IDENTITIES,
    min_images=5,
    precomputed_geo=False,   # set True if you precomputed depth/normal maps
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=(DEVICE == 'cuda'),
)
print(f'Batches per epoch: {len(loader)}')

In [ ]:
# ── Region Encoder ─────────────────────────────────────────────────────────
from models.region_encoder import FaceRegionEncoder

region_encoder = FaceRegionEncoder(
    regions=['mouth', 'eyes', 'ears', 'nose'],
    token_dim=TOKEN_DIM,
    pretrained=True,      # load ImageNet ResNet-50
    freeze_stages=3,      # freeze layer1-3, train layer4 + projections + transformer
    transformer_layers=4,
    transformer_heads=8,
    dropout=0.1,
).to(DEVICE)

trainable = sum(p.numel() for p in region_encoder.parameters() if p.requires_grad)
total     = sum(p.numel() for p in region_encoder.parameters())
print(f'Region encoder: {trainable/1e6:.1f}M trainable / {total/1e6:.1f}M total params')

In [ ]:
# ── SDXL VAE (for decoding output back to pixels) ─────────────────────────
from diffusers import AutoencoderKL

vae = AutoencoderKL.from_pretrained(
    'madebyollin/sdxl-vae-fp16-fix',
    torch_dtype=DTYPE,
).to(DEVICE)
vae.eval()
for p in vae.parameters():
    p.requires_grad_(False)
print('VAE loaded and frozen')

In [ ]:
# ── Loss functions ─────────────────────────────────────────────────────────
import torchvision.models as tv_models

# Perceptual loss: VGG-19 features
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = tv_models.vgg19(weights='IMAGENET1K_V1').features[:18].eval()
        for p in vgg.parameters():
            p.requires_grad_(False)
        self.vgg = vgg

    def forward(self, pred, target):
        # pred, target: (B, 3, H, W) in [-1, 1]
        pred   = (pred   + 1) / 2
        target = (target + 1) / 2
        return F.l1_loss(self.vgg(pred), self.vgg(target))

perceptual_loss = PerceptualLoss().to(DEVICE)

# Pixel reconstruction loss
def pixel_loss(pred, target):
    return F.l1_loss(pred, target)

print('Losses ready')

In [ ]:
# ── Optimiser ──────────────────────────────────────────────────────────────
optimiser = torch.optim.AdamW(
    region_encoder.trainable_parameters(),
    lr=LR,
    weight_decay=1e-4,
    betas=(0.9, 0.999),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimiser,
    T_max=PHASE1_EPOCHS + PHASE2_EPOCHS,
    eta_min=LR * 0.1,
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
print('Optimiser ready')

In [ ]:
# ── Helper: decode latent -> image ─────────────────────────────────────────
@torch.no_grad()
def encode_to_latent(rgb_tensor):
    """(B, 3, H, W) in [0,1] -> VAE latent (B, 4, H/8, W/8)"""
    x = rgb_tensor.to(DEVICE, dtype=DTYPE) * 2 - 1   # [0,1] -> [-1,1]
    x = F.interpolate(x, size=(512, 512), mode='bilinear', align_corners=False)
    lat = vae.encode(x).latent_dist.sample() * vae.config.scaling_factor
    return lat

@torch.no_grad()
def decode_latent(lat):
    """VAE latent -> (B, 3, H, W) in [-1, 1]"""
    return vae.decode(lat / vae.config.scaling_factor).sample

print('VAE helpers ready')

In [ ]:
# ── Phase 1: Reconstruction warmup ─────────────────────────────────────────
# Source = Target (same image). Region encoder must produce tokens that
# allow reconstructing the input. Only pixel + perceptual loss.
# Since we don't have the full U-Net here, we use a lightweight decoder
# (2-layer MLP that maps tokens -> image features) as a proxy task.

# Proxy decoder: maps [B, num_regions, TOKEN_DIM] -> [B, 3, 64, 64]
num_regions = len(['mouth', 'eyes', 'ears', 'nose'])

proxy_decoder = nn.Sequential(
    nn.Linear(num_regions * TOKEN_DIM, 1024),
    nn.GELU(),
    nn.Linear(1024, 3 * 64 * 64),
    nn.Tanh(),
).to(DEVICE)

phase1_opt = torch.optim.AdamW(
    list(region_encoder.trainable_parameters()) + list(proxy_decoder.parameters()),
    lr=LR, weight_decay=1e-4
)

print('Starting Phase 1: Reconstruction warmup')
print(f'  Epochs: {PHASE1_EPOCHS}  |  Batches/epoch: {len(loader)}')

for epoch in range(1, PHASE1_EPOCHS + 1):
    region_encoder.train()
    proxy_decoder.train()
    epoch_loss = 0.0

    pbar = tqdm(loader, desc=f'Phase1 Epoch {epoch}/{PHASE1_EPOCHS}')
    for step, batch in enumerate(pbar):
        # Use source as both input AND target (reconstruction)
        src_crops = {r: batch['source'][r].to(DEVICE) for r in ['mouth','eyes','ears','nose']}
        src_rgb   = batch['source']['rgb_full'].to(DEVICE)   # (B, 3, 224, 224)

        with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
            # Encode regions -> tokens
            tokens = region_encoder.get_tokens_for_cross_attention(src_crops)
            # (B, num_regions, TOKEN_DIM)

            # Proxy decode
            flat   = tokens.flatten(1)           # (B, num_regions * TOKEN_DIM)
            recon  = proxy_decoder(flat)         # (B, 3*64*64)
            recon  = recon.view(-1, 3, 64, 64)  # (B, 3, 64, 64)

            # Target: downsample source RGB to 64x64
            target = F.interpolate(src_rgb * 2 - 1, size=(64, 64), mode='bilinear', align_corners=False)

            loss = pixel_loss(recon, target.to(DTYPE))

        phase1_opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(phase1_opt)
        scaler.update()

        epoch_loss += loss.item()
        if step % LOG_EVERY == 0:
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg = epoch_loss / len(loader)
    print(f'Phase1 Epoch {epoch}: avg_loss={avg:.4f}')
    scheduler.step()

    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'region_encoder': region_encoder.state_dict(),
            'proxy_decoder':  proxy_decoder.state_dict(),
            'optimiser':      phase1_opt.state_dict(),
        }, f'{CHECKPOINT_DIR}/phase1_epoch{epoch:02d}.pt')
        print(f'  Saved checkpoint: phase1_epoch{epoch:02d}.pt')

# Free proxy decoder after Phase 1
del proxy_decoder, phase1_opt
gc.collect()
torch.cuda.empty_cache()
print('Phase 1 complete.')

In [ ]:
# ── Phase 2: Full face swap training ───────────────────────────────────────
# Source != Target (different images, same identity)
# Region encoder tokens injected into SDXL U-Net cross-attention
# Loss: diffusion MSE + perceptual

from diffusers import UNet2DConditionModel, DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

SDXL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'

print('Loading SDXL U-Net...')
unet = UNet2DConditionModel.from_pretrained(
    SDXL_ID, subfolder='unet', torch_dtype=DTYPE
).to(DEVICE)

# Freeze U-Net backbone, only train cross-attention projections
for name, p in unet.named_parameters():
    if 'attn2' in name:   # cross-attention layers
        p.requires_grad_(True)
    else:
        p.requires_grad_(False)

unet_trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f'U-Net: {unet_trainable/1e6:.1f}M trainable params (cross-attn only)')

noise_scheduler = DDPMScheduler.from_pretrained(SDXL_ID, subfolder='scheduler')
print('SDXL ready')

In [ ]:
# ── Phase 2 optimiser (region encoder + U-Net cross-attn) ─────────────────
phase2_opt = torch.optim.AdamW(
    region_encoder.trainable_parameters() +
    [p for p in unet.parameters() if p.requires_grad],
    lr=LR * 0.1,     # lower LR for phase 2 — fine-tuning
    weight_decay=1e-4,
)

# Null text embedding (we do text-free face swapping)
tokenizer     = CLIPTokenizer.from_pretrained(SDXL_ID, subfolder='tokenizer')
text_encoder  = CLIPTextModel.from_pretrained(SDXL_ID, subfolder='text_encoder', torch_dtype=DTYPE).to(DEVICE)

with torch.no_grad():
    null_tokens = tokenizer([''], padding='max_length',
                             max_length=tokenizer.model_max_length,
                             return_tensors='pt').to(DEVICE)
    null_embeds = text_encoder(**null_tokens).last_hidden_state   # (1, 77, 768)

del text_encoder   # free memory
gc.collect()
torch.cuda.empty_cache()
print('Phase 2 optimiser ready')

In [ ]:
# ── Phase 2 training loop ──────────────────────────────────────────────────
print('Starting Phase 2: Full face swap training')
print(f'  Epochs: {PHASE2_EPOCHS}  |  Batches/epoch: {len(loader)}')

for epoch in range(1, PHASE2_EPOCHS + 1):
    region_encoder.train()
    unet.train()
    epoch_loss = 0.0

    pbar = tqdm(loader, desc=f'Phase2 Epoch {epoch}/{PHASE2_EPOCHS}')
    for step, batch in enumerate(pbar):

        src_crops = {r: batch['source'][r].to(DEVICE) for r in ['mouth','eyes','ears','nose']}
        tgt_rgb   = batch['target']['rgb_full'].to(DEVICE)   # (B, 3, 224, 224)

        B = tgt_rgb.shape[0]

        with torch.cuda.amp.autocast(enabled=(DEVICE=='cuda')):
            # 1. Encode target image to latent
            tgt_latent = encode_to_latent(tgt_rgb)   # (B, 4, 64, 64)

            # 2. Sample noise and timestep
            noise      = torch.randn_like(tgt_latent)
            timesteps  = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                       (B,), device=DEVICE).long()
            noisy_lat  = noise_scheduler.add_noise(tgt_latent, noise, timesteps)

            # 3. Encode source regions -> tokens for cross-attention
            region_tokens = region_encoder.get_tokens_for_cross_attention(src_crops)
            # (B, num_regions, TOKEN_DIM)  e.g. (B, 4, 512)

            # Null text conditioning (we use region tokens, not text)
            encoder_hidden = null_embeds.expand(B, -1, -1)   # (B, 77, 768)

            # 4. U-Net noise prediction
            # SDXL U-Net expects added_cond_kwargs for time_ids and text_embeds
            added_cond_kwargs = {
                'text_embeds': torch.zeros(B, 1280, device=DEVICE, dtype=DTYPE),
                'time_ids':    torch.zeros(B, 6,    device=DEVICE, dtype=DTYPE),
            }

            noise_pred = unet(
                noisy_lat,
                timesteps,
                encoder_hidden_states=encoder_hidden,
                added_cond_kwargs=added_cond_kwargs,
            ).sample

            # 5. Diffusion MSE loss
            diff_loss = F.mse_loss(noise_pred, noise)

            loss = diff_loss

        phase2_opt.zero_grad()
        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(
            region_encoder.trainable_parameters(), max_norm=1.0
        )
        scaler.step(phase2_opt)
        scaler.update()

        epoch_loss += loss.item()
        if step % LOG_EVERY == 0:
            pbar.set_postfix({'diff_loss': f'{diff_loss.item():.4f}'})

    avg = epoch_loss / len(loader)
    print(f'Phase2 Epoch {epoch}: avg_loss={avg:.4f}')

    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'region_encoder': region_encoder.state_dict(),
            'unet_attn':      {k: v for k, v in unet.state_dict().items() if 'attn2' in k},
            'optimiser':      phase2_opt.state_dict(),
        }, f'{CHECKPOINT_DIR}/phase2_epoch{epoch:02d}.pt')
        print(f'  Saved checkpoint: phase2_epoch{epoch:02d}.pt')

print('Phase 2 complete. Training done!')

In [ ]:
# ── Save final model ───────────────────────────────────────────────────────
torch.save(region_encoder.state_dict(), f'{CHECKPOINT_DIR}/region_encoder_final.pt')
torch.save(
    {k: v for k, v in unet.state_dict().items() if 'attn2' in k},
    f'{CHECKPOINT_DIR}/unet_cross_attn_final.pt'
)
print(f'Saved final weights to {CHECKPOINT_DIR}/')
print('Training complete!')